# CatBoost на суммарный объём окна из восьми горизонтов

CPU. Новая модель учится прогнозировать сумму восьми будущих target_1h по сети. Это ровно тот масштаб, на котором считается bias за один прогнозный день. Модель видит только доступную к старту историю цели и шести статусов. Цель нормируется причинным профилем. На каждый исторический день используются утренние старты с 06:30 до 13:30. В train входят только окна, у которых все восемь целей известны до границы обучения.

В проверке используем доли прежнего CatBoost-бленда. Независимую оценку суммарного объёма смешиваем с Session6 на 15%, 30% или 50%, ограничивая изменение объёма окна ±2%. Один коэффициент действует на все восемь горизонтов, поэтому профиль по получасам и доли маршрутов сохраняются. Вес должен улучшить средний дневной score на обеих неделях подбора.

Сабмит допускается при выигрыше от 0.001 по средней дневной метрике на локальном test и ухудшении общей метрики не более 0.0002. Это дополнительная проверка на ранее изученном test, до полного переобучения. После допуска модель объёма переобучается на всей доступной истории завершённых окон и применяется к готовому full-history родителю. Родитель по умолчанию window_volume_parent.csv рядом с ноутбуком.


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from catboost import CatBoostRegressor
from IPython.display import display

ROOT = Path(os.environ.get('BLEND_PACKAGE', '/root/experiments/session6_blend_server'))
OOF_PATH = None
TEST_PATH = None
OUT = ROOT / 'output_window_volume_cpu'
OUT.mkdir(parents=True, exist_ok=True)

BASES = ['session6', 'rf', 'cat_rich', 'xgb_rich', 'xgb_clean', 'profile28']
GROUP_KEYS = ['cutoff', 'horizon']
KEYS = GROUP_KEYS + ['route_id']
COLUMNS = KEYS + BASES + ['y', 'level24', 'current', 'delta1h', 'delta2h', 'network_current']
THREADS = 2
SEED = 42

def read_panel(path, name, start, end):
    archived = ROOT / 'blend_inputs' / 'blend_best_output' / name
    if path is None and archived.is_file():
        path = archived
    if path is None:
        candidates = [p for p in ROOT.rglob(name) if OUT not in p.parents
                      and set(COLUMNS).issubset(pq.ParquetFile(p).schema.names)]
        if len(candidates) != 1:
            raise ValueError(f'Укажите путь явно для {name}: {candidates}')
        path = candidates[0]
    frame = pd.read_parquet(path, columns=COLUMNS)
    frame['cutoff'] = pd.to_datetime(frame.cutoff)
    expected = pd.date_range(start, end, inclusive='left') + pd.Timedelta(hours=10, minutes=30)
    actual = pd.DatetimeIndex(frame.cutoff.unique()).sort_values()
    counts = frame.groupby(GROUP_KEYS).size()
    if (not actual.equals(expected) or frame.duplicated(KEYS).any()
            or set(frame.horizon.unique()) != set(range(1, 9))
            or len(counts) != len(expected)*8 or not counts.eq(1000).all()
            or not np.isfinite(frame[COLUMNS[3:]].to_numpy()).all()
            or frame.y.lt(0).any() or frame.session6.lt(0).any()):
        raise ValueError(f'Некорректная панель: {path}')
    print(path, len(frame))
    return frame.reset_index(drop=True)

DATA = ROOT/'blend_inputs/session6'
PARENT = Path(os.environ.get('WINDOW_VOLUME_PARENT',str(ROOT/'window_volume_parent.csv')))
SIGNALS = ['target_1h']+[f'status_{i}' for i in range(1,7)]
STEP = pd.Timedelta(minutes=30)
WEIGHTS = (0.,.15,.30,.50)
CAP = .02


In [ ]:
def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-5):
        raise ValueError('Не удалось сохранить заданный объём')
    return out

def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])

def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (.2, .05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-9:
                        best = loss
                        choice = source, destination, amount, trial
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights

def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(
        log_level=np.log1p(level), current=frame.current.to_numpy() / level,
        delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level,
        log_network=np.log1p(frame.network_current.to_numpy()),
        horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(),
        shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return context.clip(-50, 50).to_numpy(dtype=np.float32), level

def metric(y, prediction):
    actual = np.asarray(y, dtype=np.float64).ravel()
    predicted = np.asarray(prediction, dtype=np.float64).ravel()
    volume = actual.sum()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError('Некорректный вход метрики')
    if volume == 0:
        return dict(rows=len(actual), WAPE=np.nan, relative_bias=np.nan, score=np.nan)
    error = predicted - actual
    wape = np.abs(error).sum() / volume
    bias = error.sum() / volume
    return dict(rows=len(actual), WAPE=wape, relative_bias=bias, score=wape + abs(bias))

def fit_meta(frame):
    matrix = shape_matrix(frame)
    weights = fit_weights(matrix, frame.y)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    model = CatBoostRegressor(depth=4, iterations=220, loss_function='MAE', learning_rate=.035,
                              l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False,
                              task_type='CPU', thread_count=THREADS)
    model.fit(features, (frame.y.to_numpy(dtype=float)-baseline)/level, sample_weight=level/level.mean())
    return model, weights

def predict_meta(frame, fitted):
    model, weights = fitted
    matrix = shape_matrix(frame)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    return with_volume(frame, baseline + np.clip(model.predict(features), -.5, .5)*level)

def score_mix(frame, shape, weight):
    prediction = with_volume(frame, (1-weight)*frame.session6.to_numpy() + weight*shape)
    daily = pd.DataFrame([dict(day=str(day.date()), **metric(group.y, prediction[group.index]))
                          for day, group in frame.groupby('cutoff')])
    summary = dict(**metric(frame.y, prediction), daily_mean=daily.score.mean(),
                   daily_p90=daily.score.quantile(.9), daily_worst=daily.score.max())
    return summary, daily, prediction


In [ ]:
def read_network(path):
    frame = pd.read_parquet(path,columns=['timestamp',*SIGNALS])
    network = frame.groupby('timestamp')[SIGNALS].sum().sort_index().astype(np.float64)
    if (not network.index.equals(pd.date_range(network.index[0],network.index[-1],freq=STEP))
            or not np.isfinite(network.to_numpy()).all() or network.lt(0).any().any()):
        raise ValueError('Некорректная история сети')
    return network

def volume_features(cutoffs):
    index = pd.DatetimeIndex(cutoffs)
    pos = network.index.get_indexer(index)
    if np.any(pos < 14*48+8):
        raise ValueError('Не хватает причинной истории')
    lags = np.array([48,96,336,672])
    historical_windows = values[pos[:,None,None]-lags[None,:,None]+np.arange(1,9)[None,None,:],0].sum(axis=2)
    prior = np.median(historical_windows,axis=1)
    if np.any(prior<=0):
        raise ValueError('Нулевой сезонный профиль')
    logs = np.log1p(values)
    today = logs[pos]
    reference = np.median(logs[pos[:,None]-lags],axis=1)
    recent = np.mean(logs[pos[:,None]-np.arange(8)],axis=1)
    typical_recent = np.mean(logs[pos[:,None,None]-lags[None,:,None]-np.arange(8)[None,None,:]],axis=2)
    typical_recent = np.median(typical_recent,axis=1)
    feat = np.column_stack([today-reference,today-logs[pos-2],today-logs[pos-4],today-logs[pos-8],
                            recent-typical_recent,np.log(prior),np.log(historical_windows/prior[:,None]),
                            index.hour*60+index.minute,index.day_of_week])
    return feat,prior

def fit_window_volume(boundary):
    mask = (network.index+8*STEP < pd.Timestamp(boundary))
    minutes = network.index.hour*60+network.index.minute
    selected = np.flatnonzero(mask & (minutes>=390) & (minutes<=810) & (np.arange(len(network))>=14*48+8))
    feat,prior = volume_features(network.index[selected])
    future = values[selected[:,None]+np.arange(1,9),0].sum(axis=1)
    if np.any(future<=0):
        raise ValueError('Нулевая сумма будущего окна')
    model = CatBoostRegressor(depth=4,iterations=240,learning_rate=.04,loss_function='MAE',
                              l2_leaf_reg=15,thread_count=4,random_seed=SEED,verbose=False,
                              task_type='CPU',allow_writing_files=False)
    model.fit(feat,np.log(future/prior),sample_weight=future/future.mean())
    print('Объём окна, цели до',boundary,'строк',len(selected),flush=True)
    return model

def volume_prediction(model,cutoffs):
    feat,prior = volume_features(cutoffs)
    pred = np.exp(np.clip(model.predict(feat),-.5,.5))*prior
    if not np.isfinite(pred).all() or np.any(pred<=0):
        raise ValueError('Некорректный прогноз объёма')
    return pd.Series(pred,index=pd.DatetimeIndex(cutoffs))

def volume_mix(frame,base,expert,weight):
    base = np.asarray(base,dtype=np.float64)
    total = pd.Series(base,index=frame.index).groupby(frame.cutoff).transform('sum').to_numpy()
    expert_at_cutoff = frame.cutoff.map(expert).to_numpy()
    if np.any(total<=0) or not np.isfinite(expert_at_cutoff).all():
        raise ValueError('Неверные объёмы для смешивания')
    return base*(1+np.clip(weight*(expert_at_cutoff/total-1),-CAP,CAP))

def evaluate(frame,baseline,expert,weights,period,fold):
    rows = []
    for weight in weights:
        forecast = volume_mix(frame,baseline,expert,weight)
        daily = pd.DataFrame([metric(group.y,forecast[group.index]) for _,group in frame.groupby('cutoff')])
        rows.append(dict(period=period,fold=fold,weight=weight,**metric(frame.y,forecast),
                         daily_mean=float(daily.score.mean()),daily_abs_bias=float(daily.relative_bias.abs().mean()),
                         daily_p90=float(daily.score.quantile(.9))))
    return rows


In [ ]:
network = read_network(DATA/'train_solo_track.parquet')
values = network.to_numpy()
oof = read_panel(OOF_PATH,'base_oof.parquet','2025-09-20','2025-10-18')
rows = []
for start,end in [('2025-10-04','2025-10-11'),('2025-10-11','2025-10-18')]:
    train = oof.loc[oof.cutoff+oof.horizon*STEP < pd.Timestamp(start)].reset_index(drop=True)
    val = oof.loc[(oof.cutoff>=start)&(oof.cutoff<end)].reset_index(drop=True)
    meta = fit_meta(train)
    volume = fit_window_volume(start)
    for period,frame in [('train_oof',train),('val',val)]:
        rows.extend(evaluate(frame,predict_meta(frame,meta),volume_prediction(volume,frame.cutoff.unique()),WEIGHTS,period,start))
    display(pd.DataFrame(rows).loc[lambda t:t.fold.eq(start)])
validation = pd.DataFrame(rows)
rank = validation.loc[validation.period.eq('val')].pivot(index='fold',columns='weight',values='daily_mean')
eligible = [w for w in WEIGHTS[1:] if rank[w].lt(rank[0.]).all()]
winner = min(eligible,key=lambda w:(rank[w].mean(),w)) if eligible else 0.
validation.to_csv(OUT/'validation_metrics.csv',index=False)
(OUT/'frozen_selection.json').write_text(json.dumps(dict(weight=winner,test_used_for_weight_selection=False),indent=2))
print('Зафиксирован вес:',winner,flush=True)


In [ ]:
test = read_panel(TEST_PATH,'test_predictions.parquet','2025-10-18','2025-11-01')
meta = fit_meta(oof)
volume = fit_window_volume('2025-10-18')
for period,frame in [('train_oof',oof),('test',test)]:
    rows.extend(evaluate(frame,predict_meta(frame,meta),volume_prediction(volume,frame.cutoff.unique()),
                         list(dict.fromkeys([0.,winner])),period,'2025-10-18'))
metrics = pd.DataFrame(rows)
metrics.to_csv(OUT/'metrics.csv',index=False)
display(metrics.loc[metrics.fold.eq('2025-10-18')])
result = metrics.loc[metrics.period.eq('test')].set_index('weight')
gain = float(result.loc[0.,'daily_mean']-result.loc[winner,'daily_mean'])
promoted = bool(winner>0 and gain>=.001 and result.loc[winner,'score']<=result.loc[0.,'score']+.0002)
(OUT/'decision.json').write_text(json.dumps(dict(promoted=promoted,weight=winner,daily_gain=gain,test_before_full_refit=True),indent=2))
print('Прошёл порог для сабмита:',promoted,flush=True)


In [ ]:
if promoted and PARENT.is_file():
    request = pd.read_parquet(DATA/'test_solo_track.parquet',columns=['id','timestamp'])
    parent = pd.read_csv(PARENT)
    if (list(parent.columns)!=['id','y_pred'] or len(parent)!=8000 or len(request)!=8000
            or not parent.id.is_unique or not request.id.is_unique or set(parent.id)!=set(request.id)
            or not np.isfinite(parent.y_pred).all() or parent.y_pred.lt(0).any()):
        raise ValueError('Некорректный родительский сабмит')
    cutoff = request.timestamp.min()-STEP
    if network.index[-1]!=cutoff or request.timestamp.nunique()!=8:
        raise ValueError('Конкурсный cutoff не совпал')
    final = fit_window_volume(cutoff+STEP)
    query = request.merge(parent,on='id',validate='one_to_one',sort=False)
    query['cutoff'] = cutoff
    future = volume_prediction(final,[cutoff])
    prediction = volume_mix(query,query.y_pred,future,winner)
    if not query.id.equals(request.id) or not np.isfinite(prediction).all() or np.any(prediction<0):
        raise ValueError('Некорректные строки сабмита')
    destination = OUT/'submission_window_volume_full_history.csv'
    query[['id']].assign(y_pred=prediction).to_csv(destination.with_suffix('.partial'),index=False)
    destination.with_suffix('.partial').replace(destination)
    final.save_model(str(OUT/'window_volume_full_history.cbm'))
    print('Сабмит:',destination,flush=True)
elif promoted:
    print('Загрузите родительский CSV:',PARENT,flush=True)
print('Результаты:',OUT,flush=True)
